# MLAAD 10% Download

Notebook version of the MLAAD 10% stratified download script. Run this after installing `huggingface_hub` and signing in with `hf auth login`.

## Imports and Configuration

This cell imports the download and progress-tracking utilities and defines the Hugging Face repository, destination folder, sampling fraction, and random seed used to create a reproducible 10 percent MLAAD subset.

In [ ]:
# Purpose: Imports the download and progress-tracking utilities and defines the Hugging Face
# repository, destination folder, sampling fraction, and random seed used to create a
# reproducible 10 percent MLAAD subset.
from collections import defaultdict
from pathlib import Path
import random
import threading
import time

from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

# Configure the source repository, local destination, and reproducible sampling settings.
REPO_ID = "mueller91/MLAAD"
REPO_TYPE = "dataset"
OUTPUT_DIR = Path.home() / "datasets" / "MLAAD_10pct"
SAMPLE_FRACTION = 0.10
SEED = 42

## Repo Listing

This cell requests the complete repository file listing in a background thread, displays elapsed progress while the request is running, propagates any API error, and reports how many files were found.

In [ ]:
# Purpose: Requests the complete repository file listing in a background thread, displays
# elapsed progress while the request is running, propagates any API error, and reports how many
# files were found.
# Create the Hugging Face API client used to inspect the dataset repository.
api = HfApi()

listing_result = {}
listing_error = {}


# Wrap repository listing so it can run in a background thread with visible progress.
def list_repo_files_with_status():
    try:
        # Uses the Hugging Face token already saved by: hf auth login
        listing_result["files"] = api.list_repo_files(
            repo_id=REPO_ID,
            repo_type=REPO_TYPE
        )
    except Exception as exc:
        listing_error["error"] = exc


# Run the potentially slow repository listing without blocking progress updates.
listing_thread = threading.Thread(target=list_repo_files_with_status)
listing_thread.start()

with tqdm(
    total=None,
    desc="Listing MLAAD repo files",
    bar_format="{desc}: {elapsed} elapsed",
) as progress:
    while listing_thread.is_alive():
        time.sleep(1)
        progress.update(1)

listing_thread.join()

if "error" in listing_error:
    raise listing_error["error"]

files = listing_result["files"]
print(f"Repo listing complete: {len(files):,} files found")

## File Filtering and Grouping

This cell keeps WAV files from the synthetic-speech portion of MLAAD and groups their repository paths by language so that the later sample represents every available language.

In [ ]:
# Purpose: Keeps WAV files from the synthetic-speech portion of MLAAD and groups their
# repository paths by language so that the later sample represents every available language.
# MLAAD structure:
# fake/<language>/<model>/audio.wav
# Retain only synthetic-speech WAV paths from the repository listing.
audio_files = [
    f for f in files
    if f.startswith("fake/") and f.lower().endswith(".wav")
]

# Group audio files by language
# Group paths by language before drawing the stratified subset.
by_language = defaultdict(list)

for file in audio_files:
    parts = file.split("/")
    if len(parts) >= 4:
        language = parts[1]
        by_language[language].append(file)

## Stratified 10% Selection

This cell uses the fixed seed to select approximately 10 percent of the available recordings independently within each language and prints the selected and available counts.

In [ ]:
# Purpose: Uses the fixed seed to select approximately 10 percent of the available recordings
# independently within each language and prints the selected and available counts.
# Use a dedicated seeded generator so the selected subset is repeatable.
rng = random.Random(SEED)

selected_audio = []

# Randomly select 10% independently from every language
# Sample the requested fraction independently within each language.
for language, language_files in sorted(by_language.items()):
    language_files = sorted(language_files)

    n_select = max(
        1,
        round(len(language_files) * SAMPLE_FRACTION)
    )

    chosen = rng.sample(language_files, n_select)
    selected_audio.extend(chosen)

    print(
        f"{language}: "
        f"{len(chosen):,} / {len(language_files):,} files"
    )

## Metadata and Support Files

This cell adds metadata for every represented model directory and the available repository documentation to the selected audio paths, producing the complete list of files to download.

In [ ]:
# Purpose: Adds metadata for every represented model directory and the available repository
# documentation to the selected audio paths, producing the complete list of files to download.
# Include meta.csv for every model directory represented
# Identify every model directory represented by the sampled recordings.
selected_model_dirs = {
    str(Path(file).parent)
    for file in selected_audio
}

# Include metadata files that exist for the represented model directories.
metadata_files = [
    f"{model_dir}/meta.csv"
    for model_dir in selected_model_dirs
    if f"{model_dir}/meta.csv" in files
]

# Also preserve repository documentation
# Preserve the available licence and repository documentation.
support_files = [
    f for f in [
        "README.md",
        "LICENSE",
        "NOTICE_NO_COMMERCIAL_USE.txt",
    ]
    if f in files
]

# Combine the sampled audio, metadata, and documentation into one download plan.
files_to_download = (
    sorted(selected_audio)
    + sorted(metadata_files)
    + support_files
)

## Summary and Manifest

This cell reports the planned download size and destination, creates the local folder, and saves a manifest of the selected WAV paths so the sampled subset can be reproduced.

In [ ]:
# Purpose: Reports the planned download size and destination, creates the local folder, and
# saves a manifest of the selected WAV paths so the sampled subset can be reproduced.
print(f"\nTotal MLAAD WAV files: {len(audio_files):,}")
print(f"Selected WAV files: {len(selected_audio):,}")
print(f"Metadata files: {len(metadata_files):,}")
print(f"Total files to download: {len(files_to_download):,}")
print(f"Destination: {OUTPUT_DIR}")

# Create the destination before writing the manifest or downloaded files.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save manifest so the exact 10% subset is reproducible
# Record the exact sampled paths so the subset can be recreated later.
manifest_path = OUTPUT_DIR / "selected_files.txt"

with manifest_path.open("w") as manifest:
    for file in selected_audio:
        manifest.write(file + "\n")

## Download Loop

This cell downloads each selected audio, metadata, and support file from Hugging Face into the destination while showing overall progress and the current repository path.

In [ ]:
# Purpose: Downloads each selected audio, metadata, and support file from Hugging Face into the
# destination while showing overall progress and the current repository path.
# Download selected files only
# Download every selected repository file while reporting the current item.
for i, filename in enumerate(tqdm(files_to_download, desc="Downloading selected files"), start=1):
    print(
        f"[{i:,}/{len(files_to_download):,}] "
        f"{filename}"
    )

    hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type=REPO_TYPE,
        local_dir=OUTPUT_DIR,
    )

## Completion

This cell confirms that the subset download has finished and prints the locations of both the downloaded files and the reproducibility manifest.

In [ ]:
# Purpose: Confirms that the subset download has finished and prints the locations of both the
# downloaded files and the reproducibility manifest.
print("\nMLAAD 10% download complete.")
print(f"Saved to: {OUTPUT_DIR}")
print(f"Subset manifest: {manifest_path}")